# Fase 3 do CRISP-DM — Preparação dos Dados

**Trabalho N1 — Ciência de Dados** · Centro Universitário Católica de Santa Catarina
Prof. Dr. Claudinei Dias (Ney) · 2026

**Ferramenta de BI:** Plotly · **Dataset:** Hotel Booking Demand

**Integrantes:** Henrique Cordeiro de Oliveira · Lucas Mendonça · Victor Kunz · Nicholas Scoz · Kauã Lucindo

---

Continuação de `01_compreensao_dos_dados.ipynb`. Lá diagnosticamos; aqui
**transformamos**. O pipeline tem seis etapas, e as três que o enunciado exige
explicitamente estão marcadas com ⭐:

| # | Etapa | |
|---|---|---|
| 1 | Tratamento de valores ausentes | ⭐ |
| 2 | Substituição de valores inconsistentes | |
| 3 | Filtragem de registros inválidos | |
| 4 | Engenharia de features | ⭐ |
| 5 | Remoção de colunas irrelevantes | ⭐ |
| 6 | Conversão de tipos e carga | |

## 0. Configuração

Mesma identidade visual do notebook anterior, para que os gráficos das duas fases
conversem entre si nos slides.

In [ ]:
!pip install -q -U "plotly>=6.0" "kaleido>=1.0"

In [ ]:
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pio.renderers.default = "colab"

SUPERFICIE       = "#fcfcfb"
TEXTO_PRIMARIO   = "#0b0b0b"
TEXTO_SECUNDARIO = "#52514e"
GRADE            = "#e5e4e0"
SERIE_1 = "#2a78d6"   # azul
SERIE_2 = "#eb6834"   # laranja

pio.templates["n1"] = go.layout.Template(
    layout=dict(
        font=dict(family="Inter, Segoe UI, sans-serif", size=13, color=TEXTO_PRIMARIO),
        title=dict(font=dict(size=17), x=0, xanchor="left"),
        paper_bgcolor=SUPERFICIE,
        plot_bgcolor=SUPERFICIE,
        colorway=[SERIE_1, SERIE_2],
        xaxis=dict(showgrid=False, linecolor=GRADE, ticks="outside", tickcolor=GRADE,
                   title=dict(font=dict(color=TEXTO_SECUNDARIO))),
        yaxis=dict(gridcolor=GRADE, zeroline=False, linecolor=GRADE,
                   title=dict(font=dict(color=TEXTO_SECUNDARIO))),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title=dict(text="")),
        margin=dict(t=70, r=30, b=60, l=70),
    )
)
pio.templates.default = "n1"

print("pandas", pd.__version__, "| plotly", plotly.__version__)

## 1. Carga dos dados brutos e fotografia do "antes"

Guardamos `df_bruto` intacto. Sem essa cópia não existe comparação "antes e
depois" — que é justamente o que o enunciado pede no resultado final.

In [ ]:
URL = ("https://raw.githubusercontent.com/mendoncaluucas/DataScience/"
       "main/data/raw/hotel_bookings.csv")

df_bruto = pd.read_csv(URL)
df = df_bruto.copy()

def perfil(frame):
    """Resumo de uma tabela, para comparar estados do pipeline."""
    return pd.Series({
        "Linhas":                len(frame),
        "Colunas":               frame.shape[1],
        "Células ausentes":      int(frame.isna().sum().sum()),
        "Colunas com ausentes":  int((frame.isna().sum() > 0).sum()),
        "Memória (MB)":          round(frame.memory_usage(deep=True).sum() / 1_000_000, 1),
    })

ANTES = perfil(df_bruto)
ANTES.to_frame("Antes")

## 2. Etapa 1 — Tratamento de valores ausentes ⭐

**Não existe uma estratégia única.** Cada coluna pede uma decisão diferente, e a
decisão depende de *por que* o dado falta — não só de quanto falta.

| Coluna | Ausente | Por que falta | Estratégia |
|---|---|---|---|
| `children` | 4 | Falha de digitação | Preencher com **0** |
| `country` | 488 | País não informado no cadastro | Categoria **"Desconhecido"** |
| `agent` | 16.340 | Reserva feita **sem** agência | Código **0** = "sem agência" |
| `company` | 112.593 | Reserva **não** é corporativa | **Remover a coluna** (etapa 5) |

O ponto conceitual: em `agent` e `company`, o valor ausente **não é um erro** — ele
carrega informação ("não houve intermediário"). Preencher com a média ali destruiria
o significado. Já em `children`, o ausente é ruído e o zero é a leitura natural.

In [ ]:
df["children"] = df["children"].fillna(0).astype(int)
df["country"]  = df["country"].fillna("Desconhecido")
df["agent"]    = df["agent"].fillna(0).astype(int)

# company continua ausente aqui — é removida na etapa 5.
restantes = df.isna().sum().loc[lambda s: s > 0]
print("Colunas ainda com ausentes:")
print(restantes if len(restantes) else "  nenhuma")

## 3. Etapa 2 — Substituição de valores inconsistentes

A coluna `meal` usa o código `SC` (*self catering* — sem refeição) e também
`Undefined`. Na documentação original do dataset os dois significam a mesma coisa:
a reserva não inclui refeição.

São **duas categorias para um único conceito** — o tipo de inconsistência que infla
contagem e quebra agrupamento. Unificamos.

In [ ]:
print("Antes:")
print(df["meal"].value_counts().to_string())

df["meal"] = df["meal"].replace({"Undefined": "SC"})

print("\nDepois:")
print(df["meal"].value_counts().to_string())

## 4. Etapa 3 — Filtragem de registros inválidos

Dois problemas que nenhum preenchimento resolve — aqui a linha inteira não descreve
uma reserva possível:

1. **Reservas sem hóspede algum** (`adults + children + babies == 0`). Uma reserva
   com zero pessoas não existe no mundo real; é registro de teste ou erro de sistema.
2. **Diária (`adr`) fora da faixa plausível** — há um valor negativo e um de
   R$ 5.400, contra uma mediana perto de R$ 95. Mantemos a faixa `0 ≤ adr ≤ 1000`.

Filtrar é uma decisão com custo: perdemos linhas. Por isso medimos quanto se perde.

In [ ]:
linhas_iniciais = len(df)

hospedes = df["adults"] + df["children"] + df["babies"]
sem_hospede = (hospedes == 0).sum()
df = df[hospedes > 0].copy()

fora_faixa = (~df["adr"].between(0, 1000)).sum()
df = df[df["adr"].between(0, 1000)].copy()

removidas = linhas_iniciais - len(df)
print(f"reservas sem hóspede .... {sem_hospede:>6}")
print(f"adr fora da faixa ....... {fora_faixa:>6}")
print(f"total removido .......... {removidas:>6}  "
      f"({removidas / linhas_iniciais * 100:.2f}% da base)")

## 5. Etapa 4 — Engenharia de features ⭐

Criar colunas que **não existiam** e que respondem melhor à pergunta de negócio que
os campos originais. Cinco novas:

| Nova coluna | Fórmula | Para que serve |
|---|---|---|
| `total_noites` | noites de semana + fim de semana | Duração real da estadia |
| `total_hospedes` | adultos + crianças + bebês | Tamanho do grupo |
| `tem_criancas` | crianças + bebês > 0 | Separa viagem em família de viagem a trabalho |
| `receita_estimada` | diária × total de noites | **Quanto se perde** quando cancela |
| `faixa_antecedencia` | `lead_time` em faixas | Torna a hipótese 1 legível no dashboard |
| `data_chegada` | ano + mês + dia reunidos | Permite análise temporal |

`receita_estimada` é a mais importante: transforma "37% cancelam" em **dinheiro**,
que é a linguagem de quem decide.

In [ ]:
df["total_noites"]   = df["stays_in_week_nights"] + df["stays_in_weekend_nights"]
df["total_hospedes"] = df["adults"] + df["children"] + df["babies"]
df["tem_criancas"]   = (df["children"] + df["babies"]) > 0
df["receita_estimada"] = (df["adr"] * df["total_noites"]).round(2)

df["faixa_antecedencia"] = pd.cut(
    df["lead_time"],
    bins=[-1, 7, 30, 90, 180, 10**6],
    labels=["Até 1 semana", "1 sem. a 1 mês", "1 a 3 meses", "3 a 6 meses", "Mais de 6 meses"],
)

# Reúne os três campos de data numa data de verdade (também é conversão de tipo).
df["data_chegada"] = pd.to_datetime(
    df["arrival_date_year"].astype(str) + "-"
    + df["arrival_date_month"] + "-"
    + df["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
)

df[["total_noites", "total_hospedes", "tem_criancas", "receita_estimada",
    "faixa_antecedencia", "data_chegada"]].head()

## 6. Etapa 5 — Remoção de colunas irrelevantes ⭐

O enunciado pede **uma** remoção justificada. Removemos quatro, e a justificativa é
diferente em cada caso — inclusive uma que é o erro mais grave que se pode cometer
nesta fase.

### `reservation_status` e `reservation_status_date` — vazamento de dados

Estas são as remoções importantes. Cruzando a coluna com o alvo:

| `reservation_status` | `is_canceled` |
|---|---|
| Check-Out | 0 · 75.166 registros |
| Canceled | 1 · 43.017 registros |
| No-Show | 1 · 1.207 registros |

A correspondência é **perfeita**: `reservation_status` não *ajuda* a prever o
cancelamento, ela **é** o cancelamento com outro nome. Mantê-la produziria um
modelo com 100% de acerto e valor zero — a informação só existe *depois* que o
desfecho aconteceu, e no momento em que a previsão seria útil ela ainda não existe.

Isso se chama **vazamento de dados** (*data leakage*), e é o tipo de erro que passa
despercebido justamente porque os resultados ficam ótimos.

`reservation_status_date` tem o mesmo defeito: é a data do desfecho.

### `company` — ausência excessiva

94,3% ausente. Os 5,7% restantes não sustentam nenhuma conclusão, e a informação
útil ("é reserva corporativa?") já está em `market_segment` e `distribution_channel`.

### `arrival_date_*` — redundância

Os três campos foram condensados em `data_chegada` na etapa anterior.

In [ ]:
COLUNAS_REMOVIDAS = {
    "reservation_status":        "vazamento — determina is_canceled com 100% de precisão",
    "reservation_status_date":   "vazamento — data do desfecho",
    "company":                   "94,3% ausente; informação já coberta por market_segment",
    "arrival_date_year":         "condensada em data_chegada",
    "arrival_date_month":        "condensada em data_chegada",
    "arrival_date_day_of_month": "condensada em data_chegada",
}

df = df.drop(columns=list(COLUNAS_REMOVIDAS))

for coluna, motivo in COLUNAS_REMOVIDAS.items():
    print(f"removida  {coluna:<26} {motivo}")

## 7. Etapa 6 — Conversão de tipos e carga

Última etapa antes de "carregar" os dados: acertar os tipos.

Colunas de texto com poucos valores distintos viram `category` — o pandas passa a
guardar um código inteiro por linha em vez de repetir a string 119 mil vezes. O
ganho de memória é grande, e é o equivalente conceitual ao que o Power Query faz ao
definir o tipo de cada coluna antes de carregar no modelo.

In [ ]:
CATEGORICAS = ["hotel", "meal", "country", "market_segment", "distribution_channel",
               "reserved_room_type", "assigned_room_type", "deposit_type", "customer_type"]

memoria_antes = df.memory_usage(deep=True).sum() / 1_000_000

for coluna in CATEGORICAS:
    df[coluna] = df[coluna].astype("category")

memoria_depois = df.memory_usage(deep=True).sum() / 1_000_000

print(f"memória antes ... {memoria_antes:6.1f} MB")
print(f"memória depois .. {memoria_depois:6.1f} MB")
print(f"redução ......... {(1 - memoria_depois / memoria_antes) * 100:5.1f}%")
df.dtypes.to_frame("tipo")

## 8. Resultado: antes × depois

A visão que o enunciado pede no resultado final.

In [ ]:
DEPOIS = perfil(df)

comparativo = pd.DataFrame({"Antes": ANTES, "Depois": DEPOIS})
comparativo["Variação"] = (DEPOIS - ANTES).map(lambda v: f"{v:+g}")
comparativo

> **Atenção ao número de colunas:** 32 antes, 32 depois. Isso é
> **coincidência**, não ausência de mudança — seis colunas saíram e seis entraram.
> Vale dizer isso na apresentação antes que alguém pergunte; as listas abaixo
> mostram exatamente o que entrou e o que saiu.

In [ ]:
def mil(valor):
    """Formata inteiro com ponto como separador de milhar."""
    return f"{int(valor):,}".replace(",", ".")

print(f"Linhas ....... {mil(ANTES['Linhas'])} -> {mil(DEPOIS['Linhas'])}")
print(f"Colunas ...... {int(ANTES['Colunas'])} -> {int(DEPOIS['Colunas'])}")
print(f"Ausentes ..... {mil(ANTES['Células ausentes'])} -> {mil(DEPOIS['Células ausentes'])}")
print(f"Memória ...... {ANTES['Memória (MB)']:.1f} MB -> {DEPOIS['Memória (MB)']:.1f} MB")
print()
print("Colunas criadas:")
for c in sorted(set(df.columns) - set(df_bruto.columns)):
    print(f"  + {c}")
print()
print("Colunas removidas:")
for c in sorted(set(df_bruto.columns) - set(df.columns)):
    print(f"  - {c}")

## 9. Dashboard exploratório

O enunciado pede um "pequeno dashboard exploratório criado com os dados já limpos,
que ajude a responder ao objetivo de negócio".

Quatro painéis, cada um respondendo a uma pergunta de quem decide — e **todos usam
colunas que só existem depois da preparação**:

1. Quando a reserva é feita, quanto ela cancela? (usa `faixa_antecedencia`)
2. A política de depósito funciona?
3. Quanto dinheiro cancela por mês? (usa `receita_estimada` e `data_chegada`)
4. Qual canal de venda cancela mais?

In [ ]:
def taxa_por(coluna):
    """Taxa de cancelamento (%) por categoria, preservando a ordem da categoria."""
    obs = df.groupby(coluna, observed=True)["is_canceled"]
    return obs.mean().mul(100).round(1), obs.size()

painel = make_subplots(
    rows=2, cols=2, vertical_spacing=0.17, horizontal_spacing=0.10,
    subplot_titles=(
        "Cancelamento por antecedência da reserva",
        "Cancelamento por tipo de depósito",
        "Receita cancelada por mês de chegada",
        "Cancelamento por segmento de mercado",
    ),
)

# --- 1. antecedência ---
taxa, _ = taxa_por("faixa_antecedencia")
painel.add_trace(go.Bar(x=list(taxa.index.astype(str)), y=taxa.values,
                        marker_color=SERIE_1, text=taxa.values,
                        texttemplate="%{text:.0f}%", textposition="outside",
                        hovertemplate="%{x}<br>%{y:.1f}% canceladas<extra></extra>",
                        showlegend=False), row=1, col=1)

# --- 2. depósito ---
taxa, _ = taxa_por("deposit_type")
painel.add_trace(go.Bar(x=list(taxa.index.astype(str)), y=taxa.values,
                        marker_color=SERIE_1, text=taxa.values,
                        texttemplate="%{text:.0f}%", textposition="outside",
                        hovertemplate="%{x}<br>%{y:.1f}% canceladas<extra></extra>",
                        showlegend=False), row=1, col=2)

# --- 3. receita cancelada por mês ---
perdida = (df[df["is_canceled"] == 1]
           .groupby(df["data_chegada"].dt.to_period("M"), observed=True)["receita_estimada"]
           .sum()
           .div(1_000_000))
painel.add_trace(go.Scatter(x=perdida.index.astype(str), y=perdida.values,
                            mode="lines+markers", line=dict(color=SERIE_2, width=2),
                            marker=dict(size=6),
                            hovertemplate="%{x}<br>R$ %{y:.2f} mi cancelados<extra></extra>",
                            showlegend=False), row=2, col=1)

# --- 4. segmento ---
taxa, tamanho = taxa_por("market_segment")
relevantes = taxa[tamanho > 1000].sort_values(ascending=False)
painel.add_trace(go.Bar(x=list(relevantes.index.astype(str)), y=relevantes.values,
                        marker_color=SERIE_1, text=relevantes.values,
                        texttemplate="%{text:.0f}%", textposition="outside",
                        hovertemplate="%{x}<br>%{y:.1f}% canceladas<extra></extra>",
                        showlegend=False), row=2, col=2)

painel.update_layout(
    height=820,
    title="Onde estão os cancelamentos — dados já preparados",
    margin=dict(t=110, r=40, b=70, l=70),
)
painel.update_yaxes(title_text="% canceladas", range=[0, 110], row=1, col=1)
painel.update_yaxes(title_text="% canceladas", range=[0, 110], row=1, col=2)
painel.update_yaxes(title_text="R$ milhões", row=2, col=1)
painel.update_yaxes(title_text="% canceladas", range=[0, 110], row=2, col=2)
painel.update_xaxes(tickangle=-25, row=1, col=1)
painel.update_xaxes(tickangle=-25, row=2, col=1)
painel.update_xaxes(tickangle=-25, row=2, col=2)
painel.show()

### Leitura do dashboard

Anotem o que **os números de vocês** mostrarem — abaixo está o que esperar, mas a
conclusão tem que sair da execução:

- **Antecedência:** a taxa sobe de forma monótona com a faixa. Confirma a hipótese 1
  do notebook anterior, agora numa forma que cabe num slide.
- **Depósito:** o resultado é contraintuitivo — `Non Refund` (não reembolsável)
  cancela *mais*, não menos. Vale discutir na apresentação: provavelmente o hotel
  exige depósito **justamente** das reservas que já considera arriscadas. É
  correlação, não causa — e reconhecer isso mostra maturidade analítica.
- **Receita cancelada:** dá a dimensão financeira do problema.
- **Segmento:** identifica o canal que mais cancela, ou seja, onde agir primeiro.

## 10. Carga — salvar o dataset preparado

Fim do pipeline. O arquivo resultante é o que alimentaria o dashboard em Dash e
qualquer modelagem posterior (Fase 4 do CRISP-DM, fora do escopo deste trabalho).

In [ ]:
from pathlib import Path

SAIDA = Path("hotel_bookings_limpo.csv")
df.to_csv(SAIDA, index=False)

tamanho = SAIDA.stat().st_size / 1_000_000
print(f"{SAIDA} salvo ({tamanho:.1f} MB) — {len(df):,} linhas x {df.shape[1]} colunas"
      .replace(",", "."))
print("Arquivo derivado — não vai versionado; esta célula o reconstrói em segundos.")

In [ ]:
# Exporta o dashboard como PNG para os slides
PASTA_PRINTS = Path("prints")
PASTA_PRINTS.mkdir(exist_ok=True)

try:
    painel.write_image(PASTA_PRINTS / "05_dashboard_exploratorio.png",
                       width=1400, height=painel.layout.height, scale=2)
    print("ok      05_dashboard_exploratorio.png")
except Exception as erro:
    print(f"falhou -> {type(erro).__name__}: {erro}")
    print("plano B: ícone de câmera na barra do gráfico")

In [ ]:
# Baixa o dataset limpo e o print (só funciona no Colab)
from google.colab import files

files.download(str(SAIDA))
files.download(str(PASTA_PRINTS / "05_dashboard_exploratorio.png"))

---

## O que esta fase entrega

| Exigência do enunciado | Onde está |
|---|---|
| Estratégia de tratamento de ausentes | Etapa 1 — quatro estratégias distintas |
| Criação de nova coluna | Etapa 4 — seis colunas novas |
| Remoção de coluna irrelevante, justificada | Etapa 5 — com o caso de vazamento |
| Visão antes × depois | Seção 8 |
| Dashboard exploratório | Seção 9 |

## Próximo passo

`app/app.py` — o mesmo dashboard como aplicação Dash, com filtros interativos.